# Review checks (2026-11-17): regex sensitivity of the instruction-reason share, and exact tests
Asked for by the peer review (Open ScholarPeer) of the paper. Reads saved P14/P15 judgements and summaries; no model runs.
1. The pre-set regex counts a bare `contradict`. Which rejections of fix lessons match only through it, and do those cite the task or only another lesson? Rule: a contradict-only reason counts as citing the instructions only if it also names the prompt, the task, a requirement or a constraint. 2. Two-sided Fisher exact tests for the paper's headline comparisons.

In [1]:
import csv, re, json
from pathlib import Path
from scipy.stats import fisher_exact
REPO = Path.cwd()
while not (REPO / "quick_runs").exists(): REPO = REPO.parent
PRE = re.compile(r"task (says|asks|states|requires|instructs)|instruction|use all|all available|contradict", re.I)
NOC = re.compile(r"task (says|asks|states|requires|instructs)|instruction|use all|all available", re.I)
TASKWORD = re.compile(r"prompt|task|requirement|constraint", re.I)
out = {}
for f, arm in [("P14", "G3o"), ("P14", "G3n"), ("P15", "Q3")]:
    R = [r for r in csv.DictReader(open(REPO / f"quick_runs/results/{f}/judgements.csv"))
         if r["arm"] == arm and r["kind"] == "fix" and r["useful"].lower() in ("false", "0")]
    pre = [r for r in R if PRE.search(r["reason"])]
    only_c = [r for r in pre if not NOC.search(r["reason"])]
    lesson_only = [r for r in only_c if not TASKWORD.search(r["reason"])]
    n_instr = len(pre) - len(lesson_only)
    out[arm] = {"rejected_fix": len(R), "pre_set": len(pre), "contradict_only": len(only_c),
                "contradict_only_lesson_not_task": len(lesson_only), "citing_instructions": n_instr,
                "share": round(n_instr / len(R), 2)}
    print(arm, out[arm])
    for r in only_c: print("   ", "LESSON-ONLY" if r in lesson_only else "task      ", r["reason"][:160])

G3o {'rejected_fix': 47, 'pre_set': 39, 'contradict_only': 3, 'contradict_only_lesson_not_task': 0, 'citing_instructions': 39, 'share': 0.83}
    task       This note encourages excluding generic labels like 'column_x', which contradicts the literal requirement of the prompt.
    task       Similar to 135, this encourages the agent to filter columns based on causal reasoning, which contradicts the specific constraint to keep all numerical columns.
    task       The task explicitly instructs to 'Keep only numerical feature columns'. Note 129 suggests manually selecting 'causal' drivers instead of blindly selecting all n
G3n {'rejected_fix': 48, 'pre_set': 38, 'contradict_only': 11, 'contradict_only_lesson_not_task': 6, 'citing_instructions': 32, 'share': 0.67}
    LESSON-ONLY This note suggests investigating and potentially removing columns like 'column_x', which contradicts Note [id 130] that explicitly warns against accidentally dr
    LESSON-ONLY This note advises excluding generic 

In [2]:
# Exact tests (two-sided Fisher), runs as the unit
tests = {"wrong shown: embedding 10/20 vs guide 5/20": (10, 5, 20), "wrong shown: Gemma guide 5/20 vs Qwen guide 1/20": (5, 1, 20),
         "passed: guide R=3 20/40 vs Qwen guide 14/40": (20, 14, 40), "passed: embedding 19/40 vs guide R=3 20/40": (19, 20, 40),
         "passed: embedding 19/40 vs all own lessons 30/40": (19, 30, 40), "passed: no memory 3/40 vs embedding 19/40": (3, 19, 40)}
pv = {k: round(fisher_exact([[a, n - a], [b, n - b]])[1], 4) for k, (a, b, n) in tests.items()}
for k, v in pv.items(): print(f"{v:.4f}  {k}")
json.dump({"instr_share": out, "fisher_p": pv}, open(REPO / "quick_runs/results/P14/review_checks.json", "w"), indent=1)

0.1908  wrong shown: embedding 10/20 vs guide 5/20
0.1818  wrong shown: Gemma guide 5/20 vs Qwen guide 1/20
0.2580  passed: guide R=3 20/40 vs Qwen guide 14/40
1.0000  passed: embedding 19/40 vs guide R=3 20/40
0.0210  passed: embedding 19/40 vs all own lessons 30/40
0.0001  passed: no memory 3/40 vs embedding 19/40
